# Actor extraction & function classification: Starling-7B

## Setup

In [ ]:
# packages

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import os
import csv
import pandas as pd
import transformers
import re

import torch
from torch import cuda, bfloat16

from langchain import PromptTemplate
from langchain.chains import LLMChain
from langchain.llms import HuggingFacePipeline

from sklearn.metrics import classification_report

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
torch.clear_autocast_cache()

In [ ]:
os.getcwd()

## Load the Model from Huggingface

In [ ]:
torch.manual_seed(0)

model_id = 'berkeley-nest/Starling-LM-7B-alpha'

device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

# begin initializing HF items, need auth token for these
model_config = transformers.AutoConfig.from_pretrained(
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    config=model_config,
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()

print(f"Model loaded on {device}")

tokenizer = transformers.AutoTokenizer.from_pretrained(
    model_id)

In [ ]:
device = torch.device('cuda')
print("GPU Name:", torch.cuda.get_device_name(device))
print("Memory Usage:", torch.cuda.memory_allocated(device) / 1024 ** 3, "GB")
print("Max Memory Usage:", torch.cuda.max_memory_allocated(device) / 1024 ** 3, "GB")

In [ ]:
torch.manual_seed(0)
generate_text = transformers.pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    pad_token_id=tokenizer.eos_token_id,
    temperature=0,  # 'randomness' of outputs, 0.0 is not possible, so use a very small number
    max_new_tokens=3000,  # max number of tokens to generate in the output
    repetition_penalty=1.1  
)

llm = HuggingFacePipeline(pipeline=generate_text)

In [ ]:
def zero_shot_prompt_messages(system_prompt, main_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": main_prompt},
    ]
    prompt = generate_text.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

def count_words(sentences_list):
    # Join the list into a single string
    if sentences_list not in [None, np.nan]:
        full_text = ' '.join(sentences_list)
        # Split the string into words and count them
        word_count = len(full_text.split())
    else:
        word_count = 0
    return word_count

In [ ]:
def generated_text_to_df(generated_text):
    # Initialize lists to store extracted data
    article_ids = []
    source_names = []
    source_types = []
    source_functions = []
    direct_quotes = []
    indirect_quotes = []

    # Join the lines into a single string
    data_str = '\n'.join(generated_text)

    # Define regex patterns to extract the necessary information
    article_id_pattern = re.compile(r'"article_id":\s*"(\d+)"')
    source_pattern = re.compile(r'\{(.*?)\}', re.DOTALL)
    source_name_pattern = re.compile(r'"source_name":\s*"([^"]+)"')
    source_type_pattern = re.compile(r'"source_type":\s*"([^"]+)"')
    source_function_pattern = re.compile(r'"source_function":\s*"([^"]+)"')
    direct_quote_pattern = re.compile(r'"sentences_direct_quote":\s*(\[[^\]]*\])')
    indirect_quote_pattern = re.compile(r'"sentences_indirect_quote":\s*(\[[^\]]*\])')

    # Split the data string into individual article sections
    # article_sections = re.split(r'\}\s*{', data_str)
    article_sections = re.split(r'(?="article_id")', data_str)

    for article_section in article_sections:
        article_id_match = article_id_pattern.search(article_section)
        if not article_id_match:
        article_id = article_id_match.group(1)

        # Find all sources within the article section
        sources_matches = source_pattern.findall(article_section)
        if not sources_matches:
            source_name = None
            source_type = None
            source_function = None
            direct_quote = None
            indirect_quote = None
            article_ids.append(article_id)
            source_names.append(source_name)
            source_types.append(source_type)
            source_functions.append(source_function)
            direct_quotes.append(direct_quote)
            indirect_quotes.append(indirect_quote)
        else:
            for source_match in sources_matches:
                source_name_match = source_name_pattern.search(source_match)
                source_type_match = source_type_pattern.search(source_match)
                source_function_match = source_function_pattern.search(source_match)
                direct_quote_match = direct_quote_pattern.search(source_match)
                indirect_quote_match = indirect_quote_pattern.search(source_match)

                article_ids.append(article_id)
                source_names.append(source_name_match.group(1) if source_name_match else None)
                source_types.append(source_type_match.group(1) if source_type_match else None)
                source_functions.append(source_function_match.group(1) if source_function_match else None)

                # Clean up the quotes to remove brackets and split into list
                if direct_quote_match:
                    direct_quotes.append(re.findall(r'"(.*?)"', direct_quote_match.group(1)))
                else:
                    direct_quotes.append([])

                if indirect_quote_match:
                    indirect_quotes.append(re.findall(r'"(.*?)"', indirect_quote_match.group(1)))
                else:
                    indirect_quotes.append([])

    # Create DataFrame
    df = pd.DataFrame({
        'article_id': article_ids,
        'source_name': source_names,
        'source_type': source_types,
        'source_function': source_functions,
        'direct_quotes': direct_quotes,
        'indirect_quotes': indirect_quotes
    })

    return df

def zero_shot_prompt_messages(system_prompt, input_prompt, main_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input_prompt + main_prompt},
    ]
    prompt = generate_text.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

def count_words(sentences_list):
    # Join the list into a single string
    if sentences_list not in [None, np.nan]:
        full_text = ' '.join(sentences_list)
        # Split the string into words and count them
        word_count = len(full_text.split())
    else:
        word_count = 0
    return word_count

In [ ]:
def generated_text_to_df(generated_text):
    # Initialize lists to store extracted data
    article_ids = []
    entity_names = []
    entity_types = []
    quotes = []

    # Join the lines into a single string
    data_str = '\n'.join(generated_text)

    # Define regex patterns to extract the necessary information
    article_id_pattern = re.compile(r'"article_id":\s*"(\d+)"')
    entity_pattern = re.compile(r'\{(.*?)\}', re.DOTALL)
    entity_name_pattern = re.compile(r'"entity_name":\s*"([^"]+)"')
    entity_type_pattern = re.compile(r'"entity_type":\s*"([^"]+)"')
    quote_pattern = re.compile(r'"quote":\s*"([^"]+)"')

    # Split the data string into individual article sections
    # article_sections = re.split(r'\}\s*{', data_str)
    article_sections = re.split(r'(?="article_id")', data_str)

    for article_section in article_sections:
        article_id_match = article_id_pattern.search(article_section)
        if not article_id_match:
        article_id = article_id_match.group(1)

        # Find all entitys within the article section
        entitys_matches = entity_pattern.findall(article_section)
        if not entitys_matches:
            entity_name = None
            entity_type = None
            quote = None
            article_ids.append(article_id)
            entity_names.append(entity_name)
            entity_types.append(entity_type)
            quotes.append(quote)
        else:
            for entity_match in entitys_matches:
                entity_name_match = entity_name_pattern.search(entity_match)
                entity_type_match = entity_type_pattern.search(entity_match)
                quote_match = quote_pattern.search(entity_match)

                article_ids.append(article_id)
                entity_names.append(entity_name_match.group(1) if entity_name_match else None)
                entity_types.append(entity_type_match.group(1) if entity_type_match else None)
                quotes.append(quote_match.group(1) if quote_match else None)

    # Create DataFrame
    df = pd.DataFrame({
        'article_id': article_ids,
        'entity_name': entity_names,
        'entity_type': entity_types,
        'quotes': quotes,
    })

    return df

# Retrieve Actor DF

In [ ]:
df = pd.read_csv('actor_analysis/reliability_actors_final_cleaned_researcher.csv',
                 sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

# change article_id to integer
df['article_id'] = df['article_id'].astype(int)
print(df.shape)

In [ ]:
df_topics = pd.read_csv('actor_analysis/reliability_topics_researcher.csv',
                        sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

df_topics['article_id'] = df_topics['article_id'].astype(int)
df_topics = df_topics[df_topics['coder'] == 'researcher'] 
df_topics = df_topics[['article_id', 'about_covid', 'actors_present']].drop_duplicates()


df = pd.merge(df, df_topics, on = 'article_id', how = 'left')
print(df.shape)

In [ ]:
articles_df = pd.read_csv('actor_analysis/final_nosarticles.csv',
                 sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

# change page_id to integer
articles_df['page_id'] = articles_df['page_id'].astype(int)

articles_df = articles_df[['page_id', 'Text']].drop_duplicates()
# rename page_id to article_id
articles_df.rename(columns = {'page_id': 'article_id'}, inplace = True)

# remove line break
articles_df['Text'] = articles_df['Text'].str.replace('[LINE_BREAK]', '\n ')

print(articles_df.shape)

In [ ]:
df = pd.merge(df, articles_df, on = 'article_id', how = 'left')
print(df.shape)

In [ ]:
# limit the df to coder researcher
df = df[df['coder'] == 'researcher']
print(df.shape)

In [ ]:
df = df[df.actor_type != 'Geopolitieke entiteit']
print(df.shape)

In [ ]:
df = df[df.about_covid == 1]
print(df.shape)

In [ ]:
df[df['actors_present'] == 0]

# if name is not none then actors_present==1
df['actors_present'] = np.where(df['actor_name'].notnull(), 1, 0)
df[df['actors_present'] == 0]

In [ ]:
# merge the two dataframes
unique_articles = df[['article_id', 'Text']].drop_duplicates()
unique_articles = unique_articles.sort_values('article_id')
# shuffle the dataframe
unique_articles = unique_articles.sample(frac=1, random_state=0).reset_index(drop=True)
print(len(unique_articles))

# Actor Extraction

In [ ]:
system_prompt = """
You are a helpful AI assistant. You will be provided with a news article in Dutch and your task is to identify and extract all direct and indirect quotes along with information about the entities who provided these quotes. The entities can be person (individuals or multiple people) or an organization/institution. 
"""
input_prompt = """
Read the following article with the ID {article_id}: {text}
"""

main_prompt = """
### Definitions:
- Direct Quotes: Verbatim statements from the entity, enclosed in quotation marks.
  - Example: "We are pleased that the government has decided to reopen the hospitality industry," said the chairman of the restaurant industry association.
- Indirect Quotes: Paraphrased statements attributed to a entity using signal words, without quotation marks.
  - Example Signal Words: volgens, beweert, zegt, meldt, verklaart, vindt dat, stelt, etc.
  - Example: The ministry announced that the restrictions will be lifted next month.

### Instructions:
1. Take time to understand the context and the statements.
2. Identify and extract all direct and indirect quotes from the article.
3. Provide the following information about the entity of each quote:
   - entity name or description: The name or description of the entity.
   - entity type: Type of the entity which can be either a person/people or an organization/instution.
4. Use the following JSON format for your output. Include one entry per quote. Do not include any additional comments or explanations in the output.

```json
{{
  "article_id": "1234567",
  "quotes": [
    {{
      "quote": "We zijn blij dat de maatregelen versoepeld worden.", zei Rutte.
      "entity_name": "Rutte",
      "entity_type": "person"
    }},
    {{
      "quote": RIVM zegt dat de besmettingscijfers in de komende weken zullen dalen.
      "entity_name": "RIVM",
      "entity_type": "organization"
    }}
  ]
}}
"""

In [ ]:
zero_shot_prompt = zero_shot_prompt_messages(system_prompt, input_prompt, main_prompt)
print(zero_shot_prompt)

In [ ]:
prompt_template = PromptTemplate(
    input_variables=["article_id", "text"],
    template=zero_shot_prompt
)

In [ ]:
chain_one = LLMChain(llm = llm, prompt = prompt_template)

In [ ]:
generated_text_zeroshot = []

In [ ]:
%%time
torch.manual_seed(0)
for index, row in unique_articles.iterrows():  
    article_id = row['article_id']
    text = row['Text']

    input_variables = {
            "article_id": article_id,
            "text": text}
    # Generate text using the chain
    generated_text = chain_one.run(input_variables)
    print(generated_text)
    generated_text_zeroshot.append(generated_text)     

## Clean and Save the Generated Data

In [ ]:
df_generated = generated_text_to_df(generated_text_zeroshot)
print(df_generated.shape)
# rename entity_name to source_name and entity_type to source_type
df_generated.rename(columns = {'entity_name': 'source_name', 'entity_type': 'source_type'}, inplace = True)

In [ ]:
df_generated['source_name'] = df_generated['source_name'].str.strip()

In [ ]:
# are there duplicates? 
duplicated_df = df_generated[df_generated.duplicated(subset=['article_id', 'source_name', 'quotes'], keep=False)]
print(len(duplicated_df))

In [ ]:
# drop duplicates
df_generated = df_generated.drop_duplicates(subset=['article_id', 'source_name', 'quotes'], keep='first')

In [ ]:
df_generated[df_generated['source_type'].isin(['news article', 'News Outlet', 'news_article', 'not applicable'])]

In [ ]:
# check if source name has the following string pieces: niet, onbekend, onbekende, niet bekend, article, artikel
df_generated[df_generated['source_name'].str.contains(r'niet|onbekend|article|artikel|N/A|specif|unknown|author|news|auteur|Unnamed', case=False, na=False)]

In [ ]:
# if source name contains these string pieces then source_name, source_type and quotes are set to None
df_generated.loc[df_generated['source_name'].str.contains(r'niet|onbekend|article|artikel|N/A|specif|unknown|author|news|auteur|unnamed', case=False, na=False), ['source_name', 'source_type', 'quotes']] = [None, None, None]

In [ ]:
duplicated_df = df_generated[df_generated.duplicated(subset=['article_id', 'source_name', 'quotes'], keep=False)]
print(len(duplicated_df))

# drop duplicates
df_generated = df_generated.drop_duplicates(subset=['article_id', 'source_name', 'quotes'], keep='first')

In [ ]:
# if source name is None then source_type is set to None
df_generated.loc[df_generated.source_name.isnull(), 'source_type'] = None

In [ ]:
# make source type lowercase, and remove leading and trailing spaces
df_generated['source_type'] = df_generated['source_type'].str.lower()
df_generated['source_type'] = df_generated['source_type'].str.strip()

In [ ]:
df_generated[~df_generated.source_type.isin(['person', 'organization']) & ~df_generated.source_type.isnull()].source_name.unique()

In [ ]:
organization_names = ['Kabinet', 'The Passion', 'Brabants Dagblad', 'het kabinet']

df_generated.loc[(df_generated['source_name'].isin(organization_names)), 'source_type'] = 'organization'

In [ ]:
# drop if source name is Nederland, Duitsland, Frankrijk en Italië make it none
locations = ['Rotterdam', 'Brazilië en Mexico', 'India', 'Iran','China', 'Peking', 'Westerse landen','Nijmegen','Nederland, Duitsland, Frankrijk en Italië',
'deze vier landen', 'Guiyang']
df_generated.loc[df_generated.source_name.isin(locations), 'source_name'] = None
df_generated.loc[(df_generated['source_name'].isnull()), 'source_type'] = None
df_generated.loc[(df_generated['source_name'].isnull()), 'quotes'] = None

duplicated_df = df_generated[df_generated.duplicated(subset=['article_id', 'source_name', 'quotes'], keep=False)]
print(len(duplicated_df))

# drop duplicates
df_generated = df_generated.drop_duplicates(subset=['article_id', 'source_name', 'quotes'], keep='first')

In [ ]:
df_generated[~df_generated.source_type.isin(['person', 'organization']) & ~df_generated.source_type.isnull()].source_type.unique()

In [ ]:
# wrong types
wrong_types = ['research', 'law', 'location', 'country']

df_generated.loc[df_generated['source_type'].isin(wrong_types), 'source_name'] = None
df_generated.loc[df_generated['source_type'].isin(wrong_types), 'quotes'] = None
df_generated.loc[df_generated['source_type'].isin(wrong_types), 'source_type'] = None

duplicated_df = df_generated[df_generated.duplicated(subset=['article_id', 'source_name', 'quotes'], keep=False)]
print(len(duplicated_df))

In [ ]:
df_generated[~df_generated.source_type.isin(['person', 'organization']) & ~df_generated.source_type.isnull()].source_name.unique()

In [ ]:
df_generated.loc[~df_generated['source_type'].isin(['person', 'organization']), 'source_type'] = 'person'

In [ ]:
duplicated_df = df_generated[df_generated.duplicated(subset=['article_id', 'source_name', 'quotes'], keep=False)]
print(len(duplicated_df))

# drop duplicates
df_generated = df_generated.drop_duplicates(subset=['article_id', 'source_name', 'quotes'], keep='first')

In [ ]:
df_generated[df_generated['source_type'].isnull()]

In [ ]:
# if source_type is person or group of people, change it to Persoon
df_generated.loc[df_generated.source_type.isin(['person']), 'source_type'] = 'Persoon'
df_generated.loc[df_generated.source_type == 'organization', 'source_type'] = 'Organisatie'

In [ ]:
df_generated['nr_words_quotes'] = df_generated['quotes'].apply(count_words)

In [ ]:
df_generated[df_generated['nr_words_quotes']==0].source_name.value_counts(dropna=False)

In [ ]:
df_generated['article_id'] = df_generated['article_id'].astype(int)
df_generated = df_generated.rename(columns={'source_name': 'actor_name', 'source_type': 'actor_type', 'source_function': 'actor_function'})

In [ ]:
# limit the article_ids of df_aggregated to the article_ids of df
print(df_generated.shape, df.shape)
df_generated = df_generated[df_generated['article_id'].isin(df['article_id'])]
df = df[df['article_id'].isin(df_generated['article_id'])]
print(df_generated.shape, df.shape)

In [ ]:
df_generated.to_csv('actor_analysis/extracted_quotes_Starling.csv', sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_NONNUMERIC)

In [ ]:
def join_unique(values):
    return ' '.join(set(values))

In [ ]:
# aggregate the df by article_id, source_name, source_type, source_function and add quotes to each other and sum nr_words_quotes
df_aggregated = df_generated.groupby(['article_id', 'actor_name', 'actor_type']).agg({'quotes': join_unique, 'nr_words_quotes': 'sum'}).reset_index()
print(df_aggregated.shape)

In [ ]:
print(len(df_aggregated.article_id.unique()))
print(len(df_generated.article_id.unique()))

In [ ]:
# are there any article_ids from df_generated that is not in df_aggregated
df_noactors = df_generated[~df_generated['article_id'].isin(df_aggregated['article_id'])]

In [ ]:
# add df_noactors to df_aggregated
df_aggregated = pd.concat([df_aggregated, df_noactors], axis=0)
print(df_aggregated.shape)
df_aggregated = df_aggregated.fillna("")

In [ ]:
df_duplicates = df_aggregated[df_aggregated.duplicated(subset=['article_id', 'actor_name', 'quotes'], keep=False)]
print(len(df_duplicates))

In [ ]:
# get unique article_ids from unique_articles
unique_article_ids = unique_articles.article_id.unique()
print(len(unique_article_ids))

# merge with df_aggregated on article_id left
df_aggregated_merged = pd.merge(unique_articles, df_aggregated, on='article_id', how='left')

In [ ]:
# save the dataframe
df_aggregated.to_csv('actor_analysis/extracted_quotes_aggregated_Starling.csv', sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_NONNUMERIC)

In [ ]:
# clean actor_name in df_aggregated and df
df_aggregated['actor_name'] = df_aggregated['actor_name'].str.strip()
df['actor_name'] = df['actor_name'].str.strip()

df_aggregated['actor_name'] = df_aggregated['actor_name'].str.replace('  ', ' ')
df['actor_name'] = df['actor_name'].str.replace('  ', ' ')

# title the first letter of each word
df_aggregated['actor_name'] = df_aggregated['actor_name'].str.title()
df['actor_name'] = df['actor_name'].str.title()

# clean actor_name in df_aggregated and df
df_aggregated['actor_name'] = df_aggregated['actor_name'].str.strip()
df['actor_name'] = df['actor_name'].str.strip()

# Categorize Actor Functions

In [ ]:
df = pd.read_csv('analyses\\NOS\\actors_NER\\actors_with_sentences_checked.csv',
                 sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

# change article_id to integer
df['article_id'] = df['article_id'].astype(int)
print(df.shape)
# drop if colnames has unnamed 
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
# keep only if directly_quoted or indirectly_quoted is 1
df = df[(df['directly_quoted'] == 1) | (df['indirectly_quoted'] == 1)]
print(df.shape)

In [ ]:
df['input_text_check'] = df['actor_name'] + ':\n ' + df['relevant_sentences_string']

In [ ]:
# change these to letters
df.loc[df.actor_function == 'NL - Nationale regering - executive / uitvoerende macht', 'actor_function'] = 'A'
df.loc[df.actor_function == 'NL - Nationaal parlement en nationale partijen – wetgevende macht', 'actor_function'] = 'A'
df.loc[df.actor_function == 'NL - Nationale regionale en lokale politieke organisaties en hun ambtenaren', 'actor_function'] = 'A'
df.loc[df.actor_function == 'NL - Nationale staatsorganisaties en hun ambtenaren', 'actor_function'] = 'A'
df.loc[df.actor_function == 'NL - Nationale koninklijke familie en haar leden', 'actor_function'] = 'A'
df.loc[df.actor_function == 'Regeringen/regeringsleiders/regeringsleden en/of andere politici in een ander land dan NL, op nationaal of lokaal niveau OF staatsorganisaties en hun ambtenaren', 'actor_function'] = 'A'
df.loc[df.actor_function == 'EU-instellingen en Internationale overheidsorganisaties (ook IGO’s) en hun leden', 'actor_function'] = 'A'
df.loc[df.actor_function == 'Nationale en internationale rechterlijke macht', 'actor_function'] = 'A'
df.loc[df.actor_function == 'Wetenschappelijke/medische organisaties en onderzoekers', 'actor_function'] = 'B'
df.loc[df.actor_function == 'Openbare en semiopenbare instellingen', 'actor_function'] = 'B'
df.loc[df.actor_function == 'Zakelijke organisaties en hun werknemers', 'actor_function'] = 'B'
df.loc[df.actor_function == 'Bekende mediapersonen (anders dan journalisten)', 'actor_function'] = 'B'
df.loc[df.actor_function == 'Journalisten anders dan de schrijver van het huidige artikel of nieuwsorganisaties anders dan de nieuwsorganisatie van het huidige artikel.', 'actor_function'] = 'B'
df.loc[df.actor_function == 'Niet-governementele organisaties (NGO), maatschappelijke organisaties, en hun leden', 'actor_function'] = 'C'
df.loc[df.actor_function == 'Religieuze instellingen en hun leden (ook gelovigen)', 'actor_function'] = 'C'
df.loc[df.actor_function == 'Publiek en leden van het publiek, publieke opiniepeilingen en hun respondenten', 'actor_function'] = 'D'

# see where actor_function is NaN
df[df['actor_function'].isna() == True]
# keep only if actor_function is not NaN
df = df[df['actor_function'].isna() == False]
print(df.shape)
# make actor_function integer

In [ ]:
# get only article_id, actor_name and relevant_sentences
df_selected = df[['article_id', 'actor_name', 'actor_function', 'input_text_corrected']].drop_duplicates()
print(df_selected.shape)

# sort by article_id, actor_name
df_selected = df_selected.sort_values(by=['article_id', 'actor_name'])
df = df.sort_values(by=['article_id', 'actor_name'])

### Prompt for actor function categorization

In [ ]:
system_prompt = """
You are a helpful AI assistant. You will receive names or descriptions of persons or organizations, along with sentences from news articles where these entities are mentioned. Your task is to read all of this information about the entity and decide which function category the entity belongs to.
"""

input_prompt = """
Read the extracted information from the news article with ID {article_id} about the actor: {actor_name}. The following text begins with the actor name and provides the sentences the actor is mentioned in the article: {input_text_corrected}.
"""

main_prompt = """
1. Carefully review the information provided.

2. For this actor, classify their function into one of the following categories:

    A. Government and politics: 
        - The Dutch government, governmental organizations, and members and representatives of these organizations.
        - Dutch parliament, parliement members and political parties and members and representatives of these organizations.
        - National, regional and local political organizations and members and representatives of these organizations.
        - Dutch state organizations, law enforcement, and members and representatives of these organizations.
        - Civil servants and other governmental officials.
        - Foreign national, regional and local politicians, political organizations and members and representatives of these organizations.
        - International governmental organizations and members and representatives of these organizations.
        - National and international judiciary.
        - Dutch royal family and its members.
        - Other national, foreign or international political figures or organizations.
        
    B. Professionals and experts:
        - Scientists, medical organizations, and researchers.
        - Public and semi-public institutions and their members.
        - Business organizations, companies and their employees.
        - Journalists and news organizations.
        - Prominent media personalities and celebrities.
        - Other professionals and experts.

    C. Civil society organizations:
        - Non-governmental organizations (NGOs), civil society organizations, and members and representatives of these organizations.
        - Interest groups, professional associations (verenigingen, beroepsverenigingen, federations), sports associations (e.g., voetbalbond, sportbonden), trade unions (vakbonden) and members and representatives of these organizations.
        - Religious institutions and their members (also believers).
        - Other social non-governmental organizations, interest groups and their representatives.

    D. Citizens and members of the public:
        - The general public, residents of cities, villages, in the Netherlands, respondents to public opinion research, public groups such as students, elderly, and other members of the public.

3. Provide the information in the JSON format below. Do not include additional information or explanations.

### Example Output (JSON format):
{{
  "article_id": "2000000",
  "actor_name": "Mark Rutte",
  "actor_function_category": "A"
}}

"""

In [ ]:
zero_shot_prompt = zero_shot_prompt_messages(system_prompt, input_prompt, main_prompt)
print(zero_shot_prompt)

In [ ]:
prompt_template = PromptTemplate(
    input_variables=["article_id", "actor_name", "input_text_corrected"],
    template=zero_shot_prompt
)

In [ ]:
chain_one = LLMChain(llm = llm, prompt = prompt_template)

In [ ]:
generated_text_zeroshot_actors = []

In [ ]:
%%time
torch.manual_seed(0)
for index, row in df_selected.iterrows():  
    article_id = row['article_id']
    actor_name = row['actor_name']
    input_text_corrected = row['input_text_corrected']
    input_variables = {
            "article_id": article_id,
            "actor_name": actor_name,
            "input_text_corrected": input_text_corrected}
    # Generate text using the chain
    generated_text = chain_one.run(input_variables)
    print(generated_text)
    generated_text_zeroshot_actors.append(generated_text)    

In [ ]:
def generated_functions_extractor(strings):
    # Initialize lists to store extracted data
    article_ids = []
    actor_names = []
    actor_functions = []

    # Define regex pattern for structured output
    assistant_output_pattern = r'<\|end_of_turn\|>GPT4 Correct Assistant:\s*({.*})'
        
    # Define regex patterns for JSON fields
    article_id_pattern = r'"article_id":\s*"(\d+)"'
    actor_name_pattern = r'"actor_name":\s*"([^"]+)"'
    actor_function_pattern = r'"actor_function_category":\s*"([^"]+)"'

    # Iterate through each string
    for string_data in strings:
        # Extract content after "<|end_of_turn|>GPT4 Correct Assistant:"
        match = re.search(assistant_output_pattern, string_data, re.DOTALL)
        if match:
            json_data = match.group(1).replace("\\", "")  # Extract and clean JSON part
        else:
            continue  # Skip this string if no match found

        article_id_match = re.search(article_id_pattern, json_data)
        actor_name_match = re.search(actor_name_pattern, json_data)
        actor_function_match = re.search(actor_function_pattern, json_data)
        
        # Extract values from regex matches
        article_id = article_id_match.group(1) if article_id_match else None
        actor_name = actor_name_match.group(1) if actor_name_match else None
        actor_function = actor_function_match.group(1) if actor_function_match else None

        # Append values to the respective lists
        article_ids.append(article_id)
        actor_names.append(actor_name)
        actor_functions.append(actor_function)

    # Create DataFrame
    df = pd.DataFrame({
        'article_id': article_ids,
        'actor_name': actor_names,
        'actor_function_category': actor_functions
    })

    return df

In [ ]:
df_generated = generated_functions_extractor(generated_text_zeroshot_actors)

In [ ]:
print(df_generated.shape)
print(df_selected.shape)

In [ ]:
# see where actor_function_category is not A B C D
df_generated[~df_generated.actor_function_category.isin(['A', 'B', 'C', 'D'])]
# if actor_function_category is not A B C D then set it to A
df_generated.loc[~df_generated.actor_function_category.isin(['A', 'B', 'C', 'D']), 'actor_function_category'] = 'A'

In [ ]:
df_selected['actor_function_starling'] = df_generated['actor_function_category']
df_selected['actor_name_starling'] = df_generated['actor_name']

In [ ]:
df_selected[df_selected['actor_name'] != df_selected['actor_name_starling']]

In [ ]:
# show the crosstab
pd.crosstab(df_selected.actor_function, df_selected.actor_function_starling)

In [ ]:
# run the classification report
print(classification_report(df_selected['actor_function'], df_selected['actor_function_starling']))

In [ ]:
# save df
df_selected.to_csv('functions_Starling.csv',
          sep = ';', encoding = 'utf-8', index = False)